In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import warnings
from colorama import Fore
import os

# Preprocessing Essentials
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline

# Regressors
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.svm import SVR

# Evaluation Metrics
from sklearn.metrics._regression import mean_absolute_error, mean_squared_error, root_mean_squared_error

<hr>

In [2]:
df=pd.read_csv('../data/preprocessed/preprocessed_data.csv')

In [3]:
df.sample()

,Unnamed: 0,age,bmi,children,charges,sex_male,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
99,99,38,19.3,0,15820.699,1,1,0,0,0,1


In [4]:
df.drop(columns='Unnamed: 0',inplace=True)

In [5]:
df.sample()

,age,bmi,children,charges,sex_male,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
538,46,28.05,1,8233.0975,0,0,0,0,1,0


<hr>

## Splitting The Data into X and y 

In [6]:
X=df.drop(columns='charges')
X

,age,bmi,children,sex_male,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
0,19,27.900,0,0,1,0,0,0,1
1,18,33.770,1,1,0,0,0,1,0
2,28,33.000,3,1,0,0,0,1,0
3,33,22.705,0,1,0,0,1,0,0
4,32,28.880,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...
1332,50,30.970,3,1,0,0,1,0,0
1333,18,31.920,0,0,0,1,0,0,0
1334,18,36.850,0,0,0,0,0,1,0
1335,21,25.800,0,0,0,0,0,0,1


In [7]:
y=df['charges']
y

0       16884.92400
1        1725.55230
2        4449.46200
3       21984.47061
4        3866.85520
           ...     
1332    10600.54830
1333     2205.98080
1334     1629.83350
1335     2007.94500
1336    29141.36030
Name: charges, Length: 1337, dtype: float64

## Splitting X & y into Training and testins Data 

In [8]:
X_train,X_test,y_train,y_test =train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
X_train.shape,y_train.shape

((1069, 9), (1069,))

<hr>

## Preprocessing step

In [60]:
preprocessing_step=ColumnTransformer([
    ('OneHotEncoder',OneHotEncoder(dtype=int,drop='if_binary'),['sex','smoker','region'])
],remainder='passthrough'
)

<hr>

## Model Selection : 

## 1-KNN Model : 

In [10]:
Knn=KNeighborsRegressor(n_neighbors=7)


In [11]:
knn_pipeline=Pipeline([
    ('Scaling',StandardScaler()),
    ('model',Knn)
])
knn_pipeline

,steps,"[('Scaling', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_neighbors,7
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30


In [12]:
knn_pipeline.fit(X_train,y_train)

,steps,"[('Scaling', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,n_neighbors,7
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30


### Making A function to evaluate Regressors : 

In [13]:
def evaluate_regressor(model,model_name:str,X_train,X_test,y_train,y_test):
    print( 'Training.........' + '\n' )
    
    print( model_name + ':' )
    print( '-'*20 )
    
    model.fit( X_train, y_train ) # Training !
    y_pred_train = model.predict( X_train )
    y_pred_test = model.predict( X_test )

    print(  )

    MAE_train = mean_absolute_error( y_train, y_pred_train )
    MSE_train = mean_squared_error( y_train, y_pred_train )
    RMSE_train = root_mean_squared_error( y_train, y_pred_train )
    
    MAE_test = mean_absolute_error( y_test, y_pred_test )
    MSE_test = mean_squared_error( y_test, y_pred_test )
    RMSE_test = root_mean_squared_error( y_test, y_pred_test )

    print( f'Mean Absolute Error of training : { MAE_train }' )
    print( Fore.CYAN + f'Mean Squared Error of training : { MSE_train }' )
    print( Fore.WHITE + f'Root Mean Squared Error of Training : { RMSE_train }' )
    print(  )

    print( Fore.WHITE + f'Mean Absolute Error of Testing : { MAE_test }' )
    print( Fore.RED  + f'Mean Squared Error of Testing : { MSE_test }' )
    print( Fore.WHITE + f'Root Mean Squared Error of Testing : { RMSE_test }' )


In [14]:
evaluate_regressor(knn_pipeline,'Knn',X_train,X_test,y_train,y_test)

Training.........

Knn:
--------------------

Mean Absolute Error of training : 2927.3883902285183
Mean Squared Error of training : 22284565.67935099
Root Mean Squared Error of Training : 4720.653098814929

Mean Absolute Error of Testing : 3604.5475384664173
Mean Squared Error of Testing : 31506598.035258573
Root Mean Squared Error of Testing : 5613.073849082923


### Finding best K parameter using Gridsearch : 

In [15]:
gs_knn=GridSearchCV(knn_pipeline,param_grid={
    'model__n_neighbors':[2,3,4,5,6,7,8]
})

In [16]:
gs_knn.fit(X_train,y_train)

,estimator,Pipeline(step...eighbors=7))])
,param_grid,"{'model__n_neighbors': [2, 3, ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [17]:
print(f'KNeighborsRegressor best n_neighbours parameter is : {gs_knn.best_params_}')

KNeighborsRegressor best n_neighbours parameter is : {'model__n_neighbors': 7}


<hr>

## 2-Linear Regression : 

In [18]:
lin_reg=LinearRegression()

In [19]:
lin_reg_pipeline=Pipeline([
    ('Scaling',StandardScaler()),
    ('model',lin_reg)
])
lin_reg_pipeline

,steps,"[('Scaling', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [20]:
lin_reg_pipeline.fit(X_train,y_train)

,steps,"[('Scaling', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


In [21]:
evaluate_regressor(lin_reg_pipeline,'LinearRegression',X_train,X_test,y_train,y_test)

Training.........

LinearRegression:
--------------------

Mean Absolute Error of training : 4181.901537775145
Mean Squared Error of training : 36979860.90472867
Root Mean Squared Error of Training : 6081.106881541277

Mean Absolute Error of Testing : 4177.0455610363215
Mean Squared Error of Testing : 35478020.67523557
Root Mean Squared Error of Testing : 5956.342894363585


<hr>

## 3-Lasso : 

In [22]:
lin_reg_L1=Lasso(alpha=0.96)

In [23]:
Lasso_pipeline=Pipeline([
    ('Scaling',StandardScaler()),
    ('model',lin_reg_L1)
])
Lasso_pipeline

,steps,"[('Scaling', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,0.96
,fit_intercept,True
,precompute,False
,copy_X,True


In [24]:
evaluate_regressor(Lasso_pipeline,'Linear Regression with L1(Lasso) Regularization ',X_train,X_test,y_train,y_test)

Training.........

Linear Regression with L1(Lasso) Regularization :
--------------------

Mean Absolute Error of training : 4181.806948442003
Mean Squared Error of training : 36979867.7725545
Root Mean Squared Error of Training : 6081.107446226756

Mean Absolute Error of Testing : 4177.230609156563
Mean Squared Error of Testing : 35484469.1606143
Root Mean Squared Error of Testing : 5956.884182239428


<hr>

## 4-Ridge :

In [25]:
lin_reg_L2=Ridge()

In [26]:
Ridge_pipeline=Pipeline([
    ('Scaling',StandardScaler()),
    ('model',lin_reg_L2)
])
Ridge_pipeline

,steps,"[('Scaling', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None


In [27]:
evaluate_regressor(Ridge_pipeline,'Linear Regression with L2(Ridge) Regularization ',X_train,X_test,y_train,y_test)

Training.........

Linear Regression with L2(Ridge) Regularization :
--------------------

Mean Absolute Error of training : 4182.934666034196
Mean Squared Error of training : 36979952.88907948
Root Mean Squared Error of Training : 6081.114444662218

Mean Absolute Error of Testing : 4179.608448218486
Mean Squared Error of Testing : 35512065.31642527
Root Mean Squared Error of Testing : 5959.200056754705


### Finding best alpha Parameter using Gridsearch 

In [28]:
np.logspace(-4,2,100)

array([1.00000000e-04, 1.14975700e-04, 1.32194115e-04, 1.51991108e-04,
       1.74752840e-04, 2.00923300e-04, 2.31012970e-04, 2.65608778e-04,
       3.05385551e-04, 3.51119173e-04, 4.03701726e-04, 4.64158883e-04,
       5.33669923e-04, 6.13590727e-04, 7.05480231e-04, 8.11130831e-04,
       9.32603347e-04, 1.07226722e-03, 1.23284674e-03, 1.41747416e-03,
       1.62975083e-03, 1.87381742e-03, 2.15443469e-03, 2.47707636e-03,
       2.84803587e-03, 3.27454916e-03, 3.76493581e-03, 4.32876128e-03,
       4.97702356e-03, 5.72236766e-03, 6.57933225e-03, 7.56463328e-03,
       8.69749003e-03, 1.00000000e-02, 1.14975700e-02, 1.32194115e-02,
       1.51991108e-02, 1.74752840e-02, 2.00923300e-02, 2.31012970e-02,
       2.65608778e-02, 3.05385551e-02, 3.51119173e-02, 4.03701726e-02,
       4.64158883e-02, 5.33669923e-02, 6.13590727e-02, 7.05480231e-02,
       8.11130831e-02, 9.32603347e-02, 1.07226722e-01, 1.23284674e-01,
       1.41747416e-01, 1.62975083e-01, 1.87381742e-01, 2.15443469e-01,
      

In [29]:
np.linspace(0.01,1,100)

array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11,
       0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22,
       0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33,
       0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44,
       0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55,
       0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66,
       0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75, 0.76, 0.77,
       0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87, 0.88,
       0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99,
       1.  ])

In [30]:
gs_Lasso=GridSearchCV(Lasso_pipeline,param_grid={
    'model__alpha':np.linspace(0.01,1,100)
})

In [31]:
gs_Lasso.fit(X_train,y_train)

c:\Users\ANDALOS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.016e+09, tolerance: 1.205e+07
  model = cd_fast.enet_coordinate_descent(
c:\Users\ANDALOS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.410e+07, tolerance: 1.189e+07
  model = cd_fast.enet_coordinate_descent(
c:\Users\ANDALOS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t

,estimator,Pipeline(step...alpha=0.96))])
,param_grid,"{'model__alpha': array([0.01, ... 1. ])}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [32]:
print(f'Linear Regression regularization with Lasso (L1) best alpha parameter is : {gs_Lasso.best_params_}')

Linear Regression regularization with Lasso (L1) best alpha parameter is : {'model__alpha': np.float64(0.9600000000000001)}


<hr>

In [33]:
svm=SVR()

In [34]:
svm_pipline=Pipeline([
    ('Scaling',StandardScaler()),
    ('model',svm)
])
svm_pipline

,steps,"[('Scaling', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0


In [35]:
evaluate_regressor(svm_pipline,'SVM',X_train,X_test,y_train,y_test)

Training.........

SVM:
--------------------

Mean Absolute Error of training : 8091.029368690181
Mean Squared Error of training : 150445469.13145477
Root Mean Squared Error of Training : 12265.621432746682

Mean Absolute Error of Testing : 9268.367379123536
Mean Squared Error of Testing : 208007448.0273024
Root Mean Squared Error of Testing : 14422.463313432363


<hr>

## Random Forest :

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
Random_forest=RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=4,
    random_state=42,min_samples_split=2
)

In [38]:
evaluate_regressor(Random_forest,'RandomForestRegressor',X_train,X_test,y_train,y_test)

Training.........

RandomForestRegressor:
--------------------

Mean Absolute Error of training : 2256.234526047204
Mean Squared Error of training : 17062644.919390988
Root Mean Squared Error of Training : 4130.695452268417

Mean Absolute Error of Testing : 2365.2586005890576
Mean Squared Error of Testing : 17677546.178559523
Root Mean Squared Error of Testing : 4204.467407241912


In [39]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

In [40]:
gs_Random_forest=GridSearchCV(Random_forest,param_grid=param_grid)

In [ ]:
gs_Random_forest.fit(X_train,y_train)

In [ ]:
gs_Random_forest.best_params_

{'max_depth': 5,
 'min_samples_leaf': 4,
 'min_samples_split': 2,
 'n_estimators': 200}

<hr>

### Trying Gridsearch to find best parameters of SVM 

In [42]:
param_grid = {
    'model__C': [0.1, 1, 10, 100],
    'model__gamma': ['scale', 0.1, 0.01],
    'model__epsilon': [0.1, 0.2, 0.5],
    'model__kernel': ['rbf']}
gs_svm=GridSearchCV(svm_pipline,param_grid=param_grid
)

In [43]:
gs_svm.fit(X_train,y_train)

,estimator,"Pipeline(step...del', SVR())])"
,param_grid,"{'model__C': [0.1, 1, ...], 'model__epsilon': [0.1, 0.2, ...], 'model__gamma': ['scale', 0.1, ...], 'model__kernel': ['rbf']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [44]:
gs_svm.best_params_

{'model__C': 100,
 'model__epsilon': 0.1,
 'model__gamma': 0.1,
 'model__kernel': 'rbf'}

In [45]:
SVM_after_gridsearch=SVR(kernel='rbf',C=100,epsilon=0.1,gamma=0.1)

In [46]:
svm_pipline_after_gridsearch=Pipeline([
    ('Scaler',StandardScaler()),
    ('model',SVM_after_gridsearch)
])

In [47]:
svm_pipline_after_gridsearch.fit(X_train,y_train)

,steps,"[('Scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,kernel,'rbf'
,degree,3
,gamma,0.1
,coef0,0.0


In [48]:
evaluate_regressor(svm_pipline_after_gridsearch,'SVM AFTER GRIDSEARCH',X_train,X_test,y_train,y_test)

Training.........

SVM AFTER GRIDSEARCH:
--------------------

Mean Absolute Error of training : 5671.152905640627
Mean Squared Error of training : 109708919.24573924
Root Mean Squared Error of Training : 10474.202558941623

Mean Absolute Error of Testing : 6762.134711382523
Mean Squared Error of Testing : 154908458.7233896
Root Mean Squared Error of Testing : 12446.222668881896


<hr>

## 6-XG boost :

In [49]:
from xgboost import XGBRegressor

In [50]:
Xgb=XGBRegressor()

In [51]:
evaluate_regressor(Xgb,'XGBoost Regressor',X_train,X_test,y_train,y_test)

Training.........

XGBoost Regressor:
--------------------

Mean Absolute Error of training : 465.31817009931893
Mean Squared Error of training : 676511.1799929816
Root Mean Squared Error of Training : 822.5029969507598

Mean Absolute Error of Testing : 2856.9950462742536
Mean Squared Error of Testing : 25594906.61873952
Root Mean Squared Error of Testing : 5059.1408973006


## Trying Gridsearch To find best Param of XGBOOST 

In [52]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

In [53]:
gs_XGB=GridSearchCV(Xgb,param_grid=param_grid)

In [54]:
gs_XGB.fit(X_train,y_train)

,estimator,"XGBRegressor(...ree=None, ...)"
,param_grid,"{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], 'n_estimators': [100, 200, ...], ...}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'reg:squarederror'


In [55]:
gs_XGB.best_params_

{'colsample_bytree': 1,
 'learning_rate': 0.05,
 'max_depth': 3,
 'n_estimators': 100,
 'subsample': 0.7}

In [56]:
XGB_after_Gridsearch=XGBRegressor(colsample_bytree=1,learning_rate=0.05,max_depth=3,n_estimators=100,subsample=0.7)

In [57]:
evaluate_regressor(XGB_after_Gridsearch,'XGBOOST After applying Gridsearch',X_train,X_test,y_train,y_test)

Training.........

XGBOOST After applying Gridsearch:
--------------------

Mean Absolute Error of training : 2296.821937193493
Mean Squared Error of training : 17299777.51351176
Root Mean Squared Error of Training : 4159.300123038942

Mean Absolute Error of Testing : 2487.2179208821713
Mean Squared Error of Testing : 17967908.580327153
Root Mean Squared Error of Testing : 4238.856989841383


<hr>

## So the best Model is RandomForest 

In [58]:
import joblib

In [59]:
joblib.dump(Random_forest,'../artifacts/randomforestmodel.pkl')

['../artifacts/randomforestmodel.pkl']

<hr>

## Applying preprocessing step on XGB and dump it again to work in Streamlit 

In [61]:
Random_forest_pipeline=Pipeline([
    ('OneHotEncoder',preprocessing_step),
    ('model',Random_forest)
])

In [62]:
data=pd.read_csv('../data/cleaned/cleaned_data.csv')

In [66]:
X_s=data.drop(columns=['charges','Unnamed: 0'])
X_s

,age,sex,bmi,children,smoker,region
0,19,female,27.900,0,yes,southwest
1,18,male,33.770,1,no,southeast
2,28,male,33.000,3,no,southeast
3,33,male,22.705,0,no,northwest
4,32,male,28.880,0,no,northwest
...,...,...,...,...,...,...
1332,50,male,30.970,3,no,northwest
1333,18,female,31.920,0,no,northeast
1334,18,female,36.850,0,no,southeast
1335,21,female,25.800,0,no,southwest


In [67]:
y_s=data['charges']
y_s

0       16884.92400
1        1725.55230
2        4449.46200
3       21984.47061
4        3866.85520
           ...     
1332    10600.54830
1333     2205.98080
1334     1629.83350
1335     2007.94500
1336    29141.36030
Name: charges, Length: 1337, dtype: float64

In [68]:
X_train_s,X_test_s,y_train_s,y_test_s =train_test_split(X_s,y_s,test_size=0.2,random_state=42)

In [69]:
Random_forest_pipeline.fit(X_train_s,y_train_s)

,steps,"[('OneHotEncoder', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('OneHotEncoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [71]:
evaluate_regressor(Random_forest_pipeline,'Random_forest_pipeline',X_train_s,X_test_s,y_train_s,y_test_s)

Training.........

Random_forest_pipeline:
--------------------

Mean Absolute Error of training : 2256.1800169443923
Mean Squared Error of training : 17062327.974152297
Root Mean Squared Error of Training : 4130.657087456219

Mean Absolute Error of Testing : 2365.2675179375315
Mean Squared Error of Testing : 17677003.419176307
Root Mean Squared Error of Testing : 4204.4028611892445


In [72]:
joblib.dump(Random_forest_pipeline,'../artifacts/randomforestpipelinemodel.pkl')

['../artifacts/randomforestpipelinemodel.pkl']